# ERA5 to PRISM Downscaling (Georgia)
Training-focused update: diagnostics, tuning, and temporal depth improvement.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
if not (ROOT / "datasets").exists() and (ROOT.parent / "datasets").exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print(f"Project root: {ROOT}")


## Data and models
- ERA5 input channels: `t2m`, `u10`, `v10`, `sp`
- PRISM target: daily high-resolution temperature
- Models: persistence, linear, CNN, ConvLSTM


In [ ]:
import json
import pandas as pd
import xarray as xr
from IPython.display import Image, Markdown, display

from datasets.prism_dataset import ERA5_PRISM_Dataset

ERA5_PATH = ROOT / "data_raw/era5_georgia_temp.nc"
PRISM_DIR = ROOT / "data_raw/prism"
EVAL_DIR = ROOT / "results/evaluation"
TRAIN_LOG_DIR = ROOT / "results/training_logs"
TUNING_DIR = ROOT / "results/tuning"

if ERA5_PATH.exists() and PRISM_DIR.exists():
    era5_ds = xr.open_dataset(ERA5_PATH)
    print("ERA5 variables:", list(era5_ds.data_vars))
    ds = ERA5_PRISM_Dataset(str(ERA5_PATH), str(PRISM_DIR), history_length=6, verbose=False)
    x, y = ds[0]
    print("Usable samples (history=6):", len(ds))
    print("Input shape [T, C, H, W]:", tuple(x.shape))
    print("Target shape [C, H, W]:", tuple(y.shape))
else:
    print("Data is missing. Run:")
    print("python data_pipeline/download_era5_georgia.py --year 2023 --month 1")
    print("python data_pipeline/download_prism.py --start-date 20230101 --days 30 --variable tmean")


## Training curves

In [ ]:
cnn_curve = TRAIN_LOG_DIR / "cnn_loss_curve.png"
convlstm_curve = TRAIN_LOG_DIR / "convlstm_loss_curve.png"
cnn_log = TRAIN_LOG_DIR / "cnn_training_log.csv"
convlstm_log = TRAIN_LOG_DIR / "convlstm_training_log.csv"

if cnn_curve.exists():
    display(Markdown("### CNN loss curve"))
    display(Image(filename=str(cnn_curve)))
if convlstm_curve.exists():
    display(Markdown("### ConvLSTM loss curve"))
    display(Image(filename=str(convlstm_curve)))

if cnn_log.exists() and convlstm_log.exists():
    cnn_df = pd.read_csv(cnn_log)
    conv_df = pd.read_csv(convlstm_log)
    latest = pd.DataFrame([
        {"model": "cnn", "best_val_loss": float(cnn_df["val_loss"].min())},
        {"model": "convlstm", "best_val_loss": float(conv_df["val_loss"].min())},
    ])
    display(Markdown("### Best validation loss from final runs"))
    display(latest)
else:
    print("Run final training first:")
    print("python training/train_downscaler.py --model cnn --history-length 6 --epochs 30 --learning-rate 5e-4 --weight-decay 0 --split-seed 42 --grad-clip 1.0 --run-name cnn")
    print("python training/train_downscaler.py --model convlstm --history-length 6 --epochs 40 --learning-rate 5e-4 --weight-decay 1e-5 --split-seed 42 --grad-clip 1.0 --run-name convlstm")


## Tuning summary

In [ ]:
summary_csv = TUNING_DIR / "tuning_summary.csv"
best_json = TUNING_DIR / "best_config.json"

if summary_csv.exists():
    tune_df = pd.read_csv(summary_csv)
    ok_df = tune_df[tune_df["status"] == "ok"].copy()
    ok_df = ok_df.sort_values("best_val_loss")
    display(Markdown("### Top tuning runs"))
    display(ok_df[["run_name", "model", "history_length", "learning_rate", "weight_decay", "best_val_loss"]].head(10))
if best_json.exists():
    best_data = json.loads(best_json.read_text())
    display(Markdown("### Best configs"))
    display(best_data)
else:
    print("Run tuning first:")
    print("python training/tune_downscaler.py --models cnn convlstm --history-lengths 3 6 --learning-rates 1e-3 5e-4 1e-4 --weight-decays 0 1e-5 --epochs 20 --split-seed 42")


## Evaluation (fair split, history=6)

In [ ]:
def latest_png(folder: Path):
    files = sorted(folder.glob("comparison_*.png"))
    if not files:
        return None
    return max(files, key=lambda p: p.stat().st_mtime)

cnn_img = latest_png(EVAL_DIR / "cnn")
convlstm_img = latest_png(EVAL_DIR / "convlstm")
summary_path = EVAL_DIR / "baselines_summary.csv"
comparison_img = EVAL_DIR / "model_comparison.png"

if cnn_img:
    display(Markdown("### CNN"))
    display(Image(filename=str(cnn_img)))
if convlstm_img:
    display(Markdown("### ConvLSTM"))
    display(Image(filename=str(convlstm_img)))
if comparison_img.exists():
    display(Markdown("### Model comparison"))
    display(Image(filename=str(comparison_img)))

if summary_path.exists():
    display(Markdown("### Metrics"))
    display(pd.read_csv(summary_path))
else:
    print("Run evaluation:")
    print("python evaluation/evaluate_model.py --models persistence linear cnn convlstm --history-length 6 --num-samples 8 --split-seed 42")


## What changed

- Training was weak mainly from short runs and unstable optimizer settings.
- After tuning and stronger training, ConvLSTM improved clearly and now leads this split.
- Temporal depth still helps only when data coverage and training stability are both handled well.
